# KuaiRand 数据理解与质量检查

## 目标

- 明确数据表粒度、主键和关联关系
- 检查日志表重复、缺失、非法值、时长和时间口径
- 输出后续分析使用的数据处理规则

原则：原始数据不改动，仅创建分析副本。


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data/raw/KuaiRand-Pure/data"

print("当前目录：", Path.cwd())
print("项目目录：", PROJECT_ROOT)
print("数据目录存在：", DATA_DIR.exists())

## 1. 数据文件与表结构


In [ ]:
files = sorted(DATA_DIR.glob("*.csv"))

for file in files:
    sample = pd.read_csv(file, nrows=5)

    print("=" * 60)
    print("表名：", file.name)
    print("字段数量：", sample.shape[1])
    print("字段：", sample.columns.tolist())

## 数据表结构总结

| 表类型 | 数据粒度 | 连接键 |
|---|---|---|
| 行为日志表 | 一次用户-视频曝光/交互 | `user_id`, `video_id` |
| 用户特征表 | 一名用户 | `user_id` |
| 视频特征表 | 一个视频 | `video_id` |

随机曝光与正常推荐的曝光机制不同，后续需分开分析。


## 2. 维度表：数据粒度与主键检查


In [ ]:
#1. 读取三张特征表
user_path = DATA_DIR / "user_features_pure.csv"
video_basic_path = DATA_DIR / "video_features_basic_pure.csv"
video_stat_path = DATA_DIR / "video_features_statistic_pure.csv"

users = pd.read_csv(user_path)
video_basic = pd.read_csv(video_basic_path)
video_stat = pd.read_csv(video_stat_path)

In [ ]:
# 2. 检查行数和唯一 ID 数量
print("用户表")
print("总行数：", len(users))
print("唯一用户数：", users["user_id"].nunique())
print("重复 user_id 数：", users["user_id"].duplicated().sum())

print("\n视频基础表")
print("总行数：", len(video_basic))
print("唯一视频数：", video_basic["video_id"].nunique())
print("重复 video_id 数：", video_basic["video_id"].duplicated().sum())

print("\n视频统计表")
print("总行数：", len(video_stat))
print("唯一视频数：", video_stat["video_id"].nunique())
print("重复 video_id 数：", video_stat["video_id"].duplicated().sum())

In [ ]:
#3. 检查两张视频表是否覆盖相同视频
basic_ids = set(video_basic["video_id"])
stat_ids = set(video_stat["video_id"])

print("基础表有、统计表没有：", len(basic_ids - stat_ids))
print("统计表有、基础表没有：", len(stat_ids - basic_ids))
print("两张表共有的视频：", len(basic_ids & stat_ids))

## 主键检查结论

- 用户表 `user_id` 唯一。
- 两张视频表 `video_id` 唯一。
- 两张视频表覆盖相同视频，可一对一合并。


## 3. 日志表：读取与基本范围检查


In [ ]:
log_random = pd.read_csv(
    DATA_DIR / "log_random_4_22_to_5_08_pure.csv"
)
log_standard_before = pd.read_csv(
    DATA_DIR / "log_standard_4_08_to_4_21_pure.csv"
)
log_standard_after = pd.read_csv(
    DATA_DIR / "log_standard_4_22_to_5_08_pure.csv"
)

raw_logs = {
    "随机曝光日志": log_random,
    "前期正常推荐日志": log_standard_before,
    "同期正常推荐日志": log_standard_after,
}


In [ ]:
for name, log in raw_logs.items():
    print("=" * 60)
    print(name)
    print("记录数：", f"{len(log):,}")
    print("用户数：", f"{log['user_id'].nunique():,}")
    print("视频数：", f"{log['video_id'].nunique():,}")
    print("日期范围：", log["date"].min(), "-", log["date"].max())
    print("is_rand 分布：", log["is_rand"].value_counts(dropna=False).to_dict())


### 日志范围结论

- `is_rand` 与日志类型一致。
- 三张日志时间窗口不同，跨表比较使用比例指标。
- 分析时以表内实际日期范围为准。


## 4. 表间覆盖关系与合并安全性


In [ ]:
user_ids = set(users["user_id"])
video_ids = set(video_basic["video_id"])

for name, log in raw_logs.items():
    unknown_users = set(log["user_id"]) - user_ids
    unknown_videos = set(log["video_id"]) - video_ids
    unmatched_user_rows = ~log["user_id"].isin(user_ids)
    unmatched_video_rows = ~log["video_id"].isin(video_ids)

    print("=" * 60)
    print(name)
    print("无法匹配的用户数：", len(unknown_users))
    print("无法匹配的视频数：", len(unknown_videos))
    print("用户无法匹配的记录数：", unmatched_user_rows.sum())
    print("视频无法匹配的记录数：", unmatched_video_rows.sum())


In [ ]:
print("关联前：", len(log_random))

merged_check = (
    log_random
    .merge(
        users[["user_id"]],
        on="user_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        video_basic[["video_id"]],
        on="video_id",
        how="left",
        validate="many_to_one"
    )
)

print("关联后：", len(merged_check))


### 表间关联结论

日志中的 `user_id` 和 `video_id` 均可匹配维度表；`many_to_one` 合并验证通过，行数不膨胀。


## 5. 重复记录检查与处理


In [ ]:
for name, log in raw_logs.items():
    exact_duplicates = log.duplicated().sum()
    user_video_duplicates = log.duplicated(
        subset=["user_id", "video_id"]
    ).sum()

    print("=" * 60)
    print(name)
    print("重复用户-视频组合：", f"{user_video_duplicates:,}")
    print("完全重复记录：", f"{exact_duplicates:,}")
    print("完全重复比例：", f"{exact_duplicates / len(log):.2%}")


In [ ]:
# 抽查随机日志中的完全重复记录。
# keep=False 会同时显示原始记录和多余副本。
exact_duplicates = (
    log_random[
        log_random.duplicated(keep=False)
    ]
    .sort_values(["user_id", "video_id", "time_ms"])
)

exact_duplicates.head(20)


In [ ]:
# 对三张同类型日志应用一致的去重规则。
clean_logs = {
    name: log.drop_duplicates().copy()
    for name, log in raw_logs.items()
}

# 为后续分析保留清晰、简短的变量名。
log_random_clean = clean_logs["随机曝光日志"]
log_standard_before_clean = clean_logs["前期正常推荐日志"]
log_standard_after_clean = clean_logs["同期正常推荐日志"]

for name in raw_logs:
    before = len(raw_logs[name])
    after = len(clean_logs[name])

    print("=" * 60)
    print(name)
    print("处理前：", f"{before:,}")
    print("处理后：", f"{after:,}")
    print("删除副本：", f"{before - after:,}")
    print("处理后完全重复数：", clean_logs[name].duplicated().sum())


### 重复记录处理结论

完全重复记录视为重复写入，每组保留一条；同一用户在不同时间重复看到同一视频视为真实曝光，保留。


## 6. 缺失值与二分类字段合法性


In [ ]:
for log_name, log in clean_logs.items():
    missing = log.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print("=" * 60)
    print(log_name)

    if missing.empty:
        print("没有缺失值")
    else:
        print(missing)

In [ ]:
binary_columns = [
    "is_click",
    "is_like",
    "is_follow",
    "is_comment",
    "is_forward",
    "is_hate",
    "long_view",
    "is_profile_enter",
    "is_rand",
]

for log_name, log in clean_logs.items():
    print("=" * 60)
    print(log_name)

    for column in binary_columns:
        values = sorted(log[column].dropna().unique())
        invalid_count = (~log[column].isin([0, 1])).sum()
        print(f"{column}: 取值={values}, 非法记录={invalid_count}")


### 字段完整性与合法性结论

三张日志无缺失值；行为标记字段均为合法 0/1；`is_rand` 与曝光机制一致。


## 7. 时长字段检查与零视频时长处理


In [ ]:
duration_columns = [
    "play_time_ms",
    "duration_ms",
    "profile_stay_time",
    "comment_stay_time"
]

for log_name, log in clean_logs.items():
    print("=" * 60)
    print(log_name)

    summary = log[duration_columns].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ).T

    display(summary)

In [ ]:
for log_name, log in clean_logs.items():
    print("=" * 60)
    print(log_name)

    for column in duration_columns:
        print(
            column,
            "负数数量：", (log[column] < 0).sum(),
            "零值比例：", f"{(log[column] == 0).mean():.2%}"
        )

    replay_count = (log["play_time_ms"] > log["duration_ms"]).sum()

    print(
        "播放时长超过视频时长：",
        replay_count,
        f"占比：{replay_count / len(log):.2%}"
    )

时长字段未发现负值。


In [ ]:
for name, log in clean_logs.items():
    zero_mask = log["duration_ms"] == 0

    print("=" * 60)
    print(name)
    print("零时长记录：", f"{zero_mask.sum():,}")
    print("记录占比：", f"{zero_mask.mean():.2%}")
    print("涉及视频：", f"{log.loc[zero_mask, 'video_id'].nunique():,}")


In [ ]:
zero_duration = (
    log_random_clean.loc[
        log_random_clean["duration_ms"] == 0,
        ["video_id", "duration_ms"]
    ]
    .merge(
        video_basic[["video_id", "video_duration"]],
        on="video_id",
        how="left"
    )
)

zero_duration["video_duration"].describe()

In [ ]:
print("成功匹配的行数：", zero_duration["video_id"].notna().sum())
print("video_duration 缺失数：", zero_duration["video_duration"].isna().sum())
print("video_duration 非缺失数：", zero_duration["video_duration"].notna().sum())

In [ ]:
valid_duration_map = (
    log_random_clean.loc[
        log_random_clean["duration_ms"] > 0
    ]
    .groupby("video_id")["duration_ms"]
    .median()
)

zero_video_ids = set(
    log_random_clean.loc[
        log_random_clean["duration_ms"] == 0,
        "video_id"
    ]
)

recoverable_ids = zero_video_ids & set(valid_duration_map.index)

print("零时长视频数：", len(zero_video_ids))
print("可从其他记录恢复的视频数：", len(recoverable_ids))

In [ ]:
all_standard_logs = pd.concat(
    [
        log_standard_before_clean,
        log_standard_after_clean
    ],
    ignore_index=True
)

standard_duration_map = (
    all_standard_logs.loc[
        all_standard_logs["duration_ms"] > 0
    ]
    .groupby("video_id")["duration_ms"]
    .median()
)

recoverable_from_standard = (
    zero_video_ids & set(standard_duration_map.index)
)

print("可从正常推荐日志恢复的视频数：",
      len(recoverable_from_standard))

In [ ]:
# 不修改原始 duration_ms；新增可用性标记和派生指标。
for name, log in clean_logs.items():
    log["duration_missing"] = (log["duration_ms"] == 0).astype("int8")
    log["play_ratio"] = pd.NA

    valid_duration = log["duration_ms"] > 0
    log.loc[valid_duration, "play_ratio"] = (
        log.loc[valid_duration, "play_time_ms"]
        / log.loc[valid_duration, "duration_ms"]
    )


### 零视频时长处理结论

`duration_ms=0` 无法从视频表或其他日志恢复。保留这些曝光与互动记录；播放比例、完播等依赖视频时长的指标中排除，并新增 `duration_missing` 标记。


## 8. 时间字段范围与口径一致性


In [ ]:
time_check = log_random_clean[
    ["date", "hourmin", "time_ms"]
].copy()

time_check["datetime_cn"] = (
    pd.to_datetime(
        time_check["time_ms"],
        unit="ms",
        utc=True
    )
    .dt.tz_convert("Asia/Shanghai")
)

time_check["date_from_timestamp_cn"] = (
    time_check["datetime_cn"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

time_check["event_hour_cn"] = time_check["datetime_cn"].dt.hour
time_check["hour_bucket"] = time_check["hourmin"] // 100

date_mismatch = (
    time_check["date"]
    != time_check["date_from_timestamp_cn"]
)
hour_mismatch = (
    time_check["hour_bucket"]
    != time_check["event_hour_cn"]
)

print("日期不一致数：", f"{date_mismatch.sum():,}")
print("日期不一致比例：", f"{date_mismatch.mean():.2%}")
print("小时分桶不一致数：", f"{hour_mismatch.sum():,}")
print("小时分桶不一致比例：", f"{hour_mismatch.mean():.2%}")


In [ ]:
time_check["date_parsed"] = pd.to_datetime(
    time_check["date"].astype(str),
    format="%Y%m%d"
)

time_check["timestamp_date"] = (
    time_check["datetime_cn"]
    .dt.tz_localize(None)
    .dt.normalize()
)

time_check["date_diff_days"] = (
    time_check["timestamp_date"]
    - time_check["date_parsed"]
).dt.days

print("日期差异分布：")
display(time_check.loc[date_mismatch, "date_diff_days"].value_counts())

print("不一致记录的时间戳小时分布：")
display(
    time_check.loc[date_mismatch, "event_hour_cn"]
    .value_counts()
    .sort_index()
)


### 时间字段处理规则

- 按日分析：使用 `date`
- 按小时分析：使用 `hourmin // 100`
- 精确排序：使用中国时区转换后的 `time_ms`
- 少量日期差异集中在 23 点，按时间口径差异处理，不删除。


## 数据质量处理总结

| 问题 | 处理方式 |
|---|---|
| 完全重复记录 | 每组保留一条 |
| 不同时间的用户-视频重复 | 保留 |
| 缺失值 | 未发现 |
| 二分类字段非法值 | 未发现 |
| 负时长 | 未发现 |
| `duration_ms=0` | 保留记录，限制播放比例类分析 |
| 时间口径差异 | 保留，按用途选择时间字段 |

数据质量满足后续指标分析要求。
